#### Demo inference notebook
This notebook provides a step by step demo of performing inference on test images based on the saved trained weights from the experiments in the paper. 
Since the files are huge, they can be downloaded from Huggingface repository.

Import relevant libraries

In [ ]:
import torch
import numpy as np
from mmdet.apis import init_detector, inference_detector
from torchvision.ops import nms
import cv2
import json
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torchvision
from glob import glob
import cv2
import os
import shutil
import json
import xml.etree.ElementTree as ET
from tqdm import tqdm
from PIL import Image
from mmdet.structures import DetDataSample
from utils import load_model, sliding_window_inference_dino, sliding_window_inference_grounding_dino, sliding_window_inference_retinanet, sliding_window_inference_yolov8, load_groundtruth_from_xml, plot_predictions

#### Follow below steps to perform inference on a test image based on best trained models

##### Step 1: Download the trained weights from hugging face repository
* **{gd/dino/rn/yolov8}**_full_dataset.pth --> Weights from the best Grounding DINO/DINO/RetinaNet/YOLOv8 model trained on *full_dataset* variant.


In [ ]:
from huggingface_hub import hf_hub_download

ckpt_path = hf_hub_download(
    repo_id="smAIL-WS/finetuned_GroundingDINO",
    filename="gd_full_dataset.pth",
    force_download=True  # avoids using a broken cached file
)

# Move it to your desired local folder
target_path = "../checkpoints/"
if not os.path.exists(target_path):
    os.makedirs(target_path)
shutil.copy(ckpt_path, target_path)

##### Step 2: Specify the paths to config file, downloaded weights and test image for prediction
* **gd_full_dataset.py** --> Configuration file for Grounding DINO (trained on *full_dataset* variant)
* **dino_full_dataset.py** --> Configuration file for DINO-DETR (trained on *full_dataset* variant)
* **rn_full_dataset.py** --> Configuration file for RetinaNet (trained on *full_dataset* variant)
* **yolov8_full_dataset.py** --> Configuration file for YOLOv8 (trained on *full_dataset* variant)

In [ ]:
config_path = 'uav_weed_detectopm/mmdetection/configs/grounding_dino/gd_full_dataset.py'
checkpoint_path = '../checkpoints/grounding_dino_full_dataset_ckpt.pth'

test_image_path = "groundtruths/images/Platte_20220520_Maize_048.png"
pred_save_path = "predictions/"

##### Step 3: First load the desired model, load the test image and call the sliding window inference function to get the predictions. Finally, save the predictions.

* **sliding_window_inference_grounding_dino()** --> Sliding window inference function for Grounding DINO
* **sliding_window_inference_dino()** --> Sliding window inference function for DINO
* **sliding_window_inference_retinanet()** --> Sliding window inference function for RetinaNet
* **sliding_window_inference_yolov8()** --> Sliding window inference function for YOLOv8

In [ ]:
# Load model
model = load_model(config_path, checkpoint_path, device='cuda:0')

img = cv2.imread(test_image_path)
img_name = test_image_path.split('/')[-1].split('.')[0]

bbox_preds_kept, scores_preds, labels_preds = sliding_window_inference_grounding_dino(model, img, img_name, window_sizes=(512, 1024), stride=256, nms_iou_threshold=0.3)
    
# Save predictions of all images in testset in a single tensor
    
torch.save(bbox_preds_kept,f'{pred_save_path}/full_pred_boxes.pt')
torch.save(scores_preds, f'{pred_save_path}/full_pred_scores.pt')
torch.save(labels_preds, f'{pred_save_path}/full_pred_labels.pt')

##### Step 4: Load and plot the predictions along with ground truth

In [ ]:
# ==== Example usage ====
pred_bboxes = torch.load('predictions/full_pred_boxes.pt')
pred_scores = torch.load('predictions/full_pred_scores.pt')
pred_labels = torch.load('predictions/full_pred_labels.pt')
gt_file   = "groundtruths/annotations/Platte_20220520_Maize_048.xml"


# Load files
gt_bboxes, gt_labels = load_groundtruth_from_xml(gt_file)

# Example
test_image_path = "groundtruths/images/Platte_20220520_Maize_048.png"

# Plot only first image’s predictions
plot_predictions(
    test_image_path,
    pred_bboxes,  
    pred_labels,  
    pred_scores,  
    gt_bboxes, 
    gt_labels,
    score_thresh=0.5   
)